# Ultra Low Complexity Noise Suppression (ULCNet - Simplified)

This notebook implements an end-to-end simplified version of the ULCNet model
based on the paper "Ultra Low Complexity Deep Learning Based Noise Suppression".

Features:

- STFT + Power Law Compression
- Two-stage Network (CRN + CNN)
- Training + Inference
- Audio Reconstruction


In [7]:

# Install dependencies (run once)
!pip install torch torchaudio librosa soundfile tqdm


In [8]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from tqdm import tqdm
import os


## STFT and Power-Law Compression


In [ ]:

def power_compress(x, alpha=0.3):
    return torch.sign(x) * torch.abs(x) ** alpha

def power_decompress(x, alpha=0.3):
    return torch.sign(x) * torch.abs(x) ** (1/alpha)


def compute_stft(wav, n_fft=512, hop=256):
    return torch.stft(
        wav, n_fft=n_fft, hop_length=hop,
        win_length=n_fft, return_complex=True
    )


def compute_istft(spec, n_fft=512, hop=256):
    return torch.istft(
        spec, n_fft=n_fft, hop_length=hop,
        win_length=n_fft
    )


## Stage 1: CRN (Magnitude Mask)


In [ ]:

class CRNStage(nn.Module):
    def __init__(self, freq_bins=257):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, (1,3), padding=(0,1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, (1,3), padding=(0,1)),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, (1,3), padding=(0,1)),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.gru = nn.GRU(
            input_size=128*freq_bins,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Sequential(
            nn.Linear(512, freq_bins),
            nn.Sigmoid()
        )


    def forward(self, x):
        # x: [B,1,T,F]
        B,C,T,F = x.shape

        x = self.conv(x)

        x = x.permute(0,2,1,3)
        x = x.reshape(B, T, -1)

        x,_ = self.gru(x)

        mask = self.fc(x)

        return mask


## Stage 2: CNN (Complex Mask)


In [ ]:

class CNNStage(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(2, 32, (1,3), padding=(0,1)),
            nn.ReLU(),

            nn.Conv2d(32, 32, (1,3), padding=(0,1)),
            nn.ReLU(),

            nn.Conv2d(32, 2, 1)
        )


    def forward(self, x):
        return self.net(x)


## Full ULCNet Model


In [ ]:

class ULCNet(nn.Module):
    def __init__(self, freq_bins=257, alpha=0.3):
        super().__init__()

        self.alpha = alpha

        self.stage1 = CRNStage(freq_bins)
        self.stage2 = CNNStage()


    def forward(self, mag, phase):

        # Stage 1
        mag = mag.unsqueeze(1)
        mask = self.stage1(mag)

        # Intermediate features
        yr = mask * torch.cos(phase)
        yi = mask * torch.sin(phase)

        feat = torch.stack([yr, yi], dim=1)

        # Stage 2
        cmask = self.stage2(feat)

        mr = cmask[:,0]
        mi = cmask[:,1]

        return mr, mi


## Dataset Loader (DNS-Style Simulation)


In [ ]:

class SimpleDataset(torch.utils.data.Dataset):

    def __init__(self, clean_files, noise_files, sr=16000):
        self.clean = clean_files
        self.noise = noise_files
        self.sr = sr


    def __len__(self):
        return len(self.clean)


    def __getitem__(self, idx):

        c,_ = librosa.load(self.clean[idx], sr=self.sr)
        n,_ = librosa.load(
            np.random.choice(self.noise),
            sr=self.sr
        )

        L = min(len(c), len(n))
        c = c[:L]
        n = n[:L]

        snr = np.random.uniform(-5,15)
        n = n * 10**(-snr/20)

        mix = c + n

        return torch.tensor(mix), torch.tensor(c)


## Training Loop


In [ ]:

def train(model, loader, optimizer, device):

    model.train()
    total = 0

    for mix, clean in tqdm(loader):

        mix = mix.to(device)
        clean = clean.to(device)

        X = compute_stft(mix)
        S = compute_stft(clean)

        Xr = power_compress(X.real, model.alpha)
        Xi = power_compress(X.imag, model.alpha)

        mag = torch.sqrt(Xr**2 + Xi**2)
        phase = torch.atan2(Xi, Xr)

        Sr = power_compress(S.real, model.alpha)
        Si = power_compress(S.imag, model.alpha)

        mr, mi = model(mag, phase)

        est_r = mr * Xr - mi * Xi
        est_i = mr * Xi + mi * Xr

        loss = F.mse_loss(est_r, Sr) + F.mse_loss(est_i, Si)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    return total/len(loader)


## Inference / Enhancement


In [ ]:

def enhance(model, wav, device):

    model.eval()

    with torch.no_grad():

        wav = wav.to(device)

        X = compute_stft(wav)

        Xr = power_compress(X.real, model.alpha)
        Xi = power_compress(X.imag, model.alpha)

        mag = torch.sqrt(Xr**2 + Xi**2)
        phase = torch.atan2(Xi, Xr)

        mr, mi = model(mag.unsqueeze(0), phase.unsqueeze(0))

        est_r = mr[0]*Xr - mi[0]*Xi
        est_i = mr[0]*Xi + mi[0]*Xr

        est_r = power_decompress(est_r, model.alpha)
        est_i = power_decompress(est_i, model.alpha)

        spec = torch.complex(est_r, est_i)

        out = compute_istft(spec)

    return out.cpu().numpy()


## Example Usage


In [ ]:
# training params
clean_dir = "clean_trainset_wav"
noisy_dir = "noisy_trainset_wav"

# testing params
test_file = "app2_data/noisy/p232_003.wav"
output_location = "enhanced_output.wav"

In [ ]:

# Example (requires your own wav files)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ULCNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=4e-4)

# Prepare dataset
clean_files = []
noise_files = []

# clean files append
for filename in os.listdir(clean_dir):
    if filename.endswith(".wav"):
        clean_files.append(os.path.join(clean_dir, filename))
# noise files append
for filename in os.listdir(noisy_dir):
    if filename.endswith(".wav"):
        noise_files.append(os.path.join(noisy_dir, filename))
    
    
dataset = SimpleDataset(clean_files, noise_files)
loader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True)

for epoch in range(10):
    loss = train(model, loader, opt, device)
    print("Epoch", epoch, loss)

# Enhance
wav,_ = librosa.load(test_file, sr=16000)
out = enhance(model, torch.tensor(wav), device)
sf.write(output_location, out, 16000)


## Notes

This is a faithful but simplified implementation for learning and experimentation.

For production, full subband GRUs and channelwise reorientation should be added.
